In [9]:
import torch
import transformers
print("Torch version:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

Torch version: 2.5.1+cu121
CUDA disponível: True


In [10]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import random

# Caminhos para os dados
DATASET_PATH = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\cell_images"
INFECTED_PATH = os.path.join(DATASET_PATH, "Parasitized")  # Células infectadas
UNINFECTED_PATH = os.path.join(DATASET_PATH, "Uninfected")  # Células saudáveis

# Transformações de imagem para Swin Transformer
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Tamanho esperado pelo Swin Transformer
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Criar dataset personalizado
class MalariaDataset(Dataset):
    def __init__(self, data_path, transform=None):
        self.data_path = data_path
        self.transform = transform
        self.images = []
        self.labels = []
        
        # Carregar imagens infectadas (1)
        for img_name in os.listdir(INFECTED_PATH):
            self.images.append(os.path.join(INFECTED_PATH, img_name))
            self.labels.append(1)
        
        # Carregar imagens não infectadas (0)
        for img_name in os.listdir(UNINFECTED_PATH):
            self.images.append(os.path.join(UNINFECTED_PATH, img_name))
            self.labels.append(0)

        # Embaralhar os dados
        temp = list(zip(self.images, self.labels))
        random.shuffle(temp)
        self.images, self.labels = zip(*temp)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = Image.open(self.images[idx]).convert("RGB")
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.long)

# Criar instâncias do dataset
dataset = MalariaDataset(DATASET_PATH, transform=transform)

# Dividir em treino e teste (80% treino, 20% teste)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Tamanho do dataset: {len(dataset)}")
print(f"Imagens de treino: {len(train_dataset)}, Imagens de teste: {len(test_dataset)}")


Tamanho do dataset: 27558
Imagens de treino: 22046, Imagens de teste: 5512


In [11]:
from transformers import SwinForImageClassification

# Carregar modelo Swin Transformer pré-treinado
model_name = "microsoft/swin-tiny-patch4-window7-224"
model = SwinForImageClassification.from_pretrained(model_name, num_labels=2, ignore_mismatched_sizes=True)

# Usar GPU se disponível
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


Some weights of SwinForImageClassification were not initialized from the model checkpoint at microsoft/swin-tiny-patch4-window7-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


SwinForImageClassification(
  (swin): SwinModel(
    (embeddings): SwinEmbeddings(
      (patch_embeddings): SwinPatchEmbeddings(
        (projection): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      )
      (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): SwinEncoder(
      (layers): ModuleList(
        (0): SwinStage(
          (blocks): ModuleList(
            (0): SwinLayer(
              (layernorm_before): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
              (attention): SwinAttention(
                (self): SwinSelfAttention(
                  (query): Linear(in_features=96, out_features=96, bias=True)
                  (key): Linear(in_features=96, out_features=96, bias=True)
                  (value): Linear(in_features=96, out_features=96, bias=True)
                  (dropout): Dropout(p=0.0, inplace=False)
                )
                (output): SwinSelfOutput(
        

In [12]:
import os
from PIL import Image

DATASET_PATH = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\cell_images"

# Listar arquivos que podem ser problemáticos
for folder in ["Parasitized", "Uninfected"]:
    folder_path = os.path.join(DATASET_PATH, folder)
    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)
        try:
            with Image.open(file_path) as img:
                img.verify()  # Verifica se a imagem é válida
        except Exception as e:
            print(f"Erro no arquivo: {file_path} - {str(e)}")


In [13]:
for folder in ["Parasitized", "Uninfected"]:
    folder_path = os.path.join(DATASET_PATH, folder)
    for file in os.listdir(folder_path):
        ext = os.path.splitext(file)[-1].lower()
        if ext not in [".png", ".jpg", ".jpeg"]:
            print(f"Arquivo inválido detectado: {file}")

### Treinando o modelo

In [14]:
from torchvision.models.swin_transformer import swin_t
import torch.nn as nn
import torchvision.models as models

def get_swin_model():
    model = swin_t(weights=models.Swin_T_Weights.IMAGENET1K_V1)
    num_features = model.head.in_features
    model.head = nn.Linear(num_features, 2)  # 2 classes: com e sem malária
    return model


In [15]:

def train(model, train_loader, val_loader, epochs=5, device='cuda' if torch.cuda.is_available() else 'cpu', model_name='SwinTransformer', fold=0):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

    log_file = "kfold_epoch_results.csv"
    results_list = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        current_lr = optimizer.param_groups[0]['lr']

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        scheduler.step()  #  atualiza o lr se necessário

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}, LR: {current_lr:.6f}")

        # Avaliação por época (validação a cada época)
        model.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        accuracy = 100 * np.mean(np.array(all_preds) == np.array(all_labels))
        precision = precision_score(all_labels, all_preds, zero_division=0)
        recall = recall_score(all_labels, all_preds, zero_division=0)
        f1 = f1_score(all_labels, all_preds, zero_division=0)
        cm = confusion_matrix(all_labels, all_preds)
        tn, fp, fn, tp = cm.ravel()

        results_list.append({
            "fold": fold,
            "epoch": epoch + 1,
            "model": model_name,
            "loss": running_loss / len(train_loader),
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "true_positives": tp,
            "false_positives": fp,
            "true_negatives": tn,
            "false_negatives": fn,
            "learning_rate": current_lr  # log do lr
        })

    torch.save(model.state_dict(), f"swin_fold{fold}.pth")

    results_df = pd.DataFrame(results_list)
    if os.path.exists(log_file):
        results_df.to_csv(log_file, mode='a', header=False, index=False)
    else:
        results_df.to_csv(log_file, index=False)

    '''
# K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f"\n----- Fold {fold + 1} -----")

    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)

    model = get_swin_model()
    model_name = f"SwinTransformer_Fold{fold+1}"
    train(model, train_loader, val_loader, model_name=model_name, epochs=15)
    '''


In [17]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from torch.utils.data import Subset, DataLoader

# 1. Extrai os rótulos do dataset original
labels = [dataset[i][1] for i in range(len(dataset))]

# 2. Divide em 80% treino+validação e 20% teste
train_val_indices, test_indices = train_test_split(
    list(range(len(labels))),
    test_size=0.2,
    stratify=labels,
    random_state=42
)

# 3. Cria os Subsets
train_val_dataset = Subset(dataset, train_val_indices)
test_dataset = Subset(dataset, test_indices)

# 4. Cria o DataLoader do conjunto de teste (fixo)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 5. Prepara os rótulos de treino/validação para o StratifiedKFold
train_val_labels = [dataset[i][1] for i in train_val_indices]

# 6. Configura o K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 7. Inicia o loop de treinamento por fold
for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(train_val_dataset)), train_val_labels)):
    print(f"\n----- Fold {fold + 1} -----")

    train_subset = Subset(train_val_dataset, train_idx)
    val_subset = Subset(train_val_dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)

    model = get_swin_model()
    model_name = f"SwinTransformer_Fold{fold+1}"
    train(model, train_loader, val_loader, model_name=model_name, epochs=20, fold=fold+1)



----- Fold 1 -----
Epoch [1/20], Loss: 0.1390, LR: 0.000100
Epoch [2/20], Loss: 0.1013, LR: 0.000100
Epoch [3/20], Loss: 0.0850, LR: 0.000100
Epoch [4/20], Loss: 0.0718, LR: 0.000100
Epoch [5/20], Loss: 0.0616, LR: 0.000100
Epoch [6/20], Loss: 0.0532, LR: 0.000100
Epoch [7/20], Loss: 0.0449, LR: 0.000100
Epoch [8/20], Loss: 0.0360, LR: 0.000100
Epoch [9/20], Loss: 0.0306, LR: 0.000100
Epoch [10/20], Loss: 0.0298, LR: 0.000100
Epoch [11/20], Loss: 0.0127, LR: 0.000010
Epoch [12/20], Loss: 0.0076, LR: 0.000010
Epoch [13/20], Loss: 0.0069, LR: 0.000010
Epoch [14/20], Loss: 0.0048, LR: 0.000010
Epoch [15/20], Loss: 0.0043, LR: 0.000010
Epoch [16/20], Loss: 0.0031, LR: 0.000010
Epoch [17/20], Loss: 0.0031, LR: 0.000010
Epoch [18/20], Loss: 0.0035, LR: 0.000010
Epoch [19/20], Loss: 0.0026, LR: 0.000010
Epoch [20/20], Loss: 0.0018, LR: 0.000010

----- Fold 2 -----
Epoch [1/20], Loss: 0.1394, LR: 0.000100
Epoch [2/20], Loss: 0.0985, LR: 0.000100
Epoch [3/20], Loss: 0.0817, LR: 0.000100
Epoch 

### testando no dataset de teste

In [22]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report
import pandas as pd
import numpy as np
import torch.nn as nn
import os

criterion = nn.CrossEntropyLoss()

def evaluate(model, test_loader, device='cuda' if torch.cuda.is_available() else 'cpu', fold=None, model_name='SwinTransformer', epoch=None, log_file="kfold_epoch_results.csv"):
    model.to(device)
    model.eval()

    all_preds = []
    all_labels = []
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / len(test_loader)
    accuracy = 100 * np.mean(np.array(all_preds) == np.array(all_labels))
    precision = precision_score(all_labels, all_preds, zero_division=0, average='binary')
    recall = recall_score(all_labels, all_preds, zero_division=0, average='binary')
    f1 = f1_score(all_labels, all_preds, zero_division=0, average='binary')

    cm = confusion_matrix(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()

    print(f"\nAvaliação do modelo {model_name} - Fold {fold} - Epoch {epoch}")
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
    print(f"TP: {tp} | FP: {fp} | TN: {tn} | FN: {fn}")

    report = classification_report(all_labels, all_preds, digits=4)
    print("\n Classification Report:\n")
    print(report)

    result = {
        "fold": fold,
        "epoch": epoch,
        "model": model_name,
        "loss": avg_loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "true_positives": tp,
        "false_positives": fp,
        "true_negatives": tn,
        "false_negatives": fn
    }

    results_df = pd.DataFrame([result])
    if os.path.exists(log_file):
        results_df.to_csv(log_file, mode='a', header=False, index=False)
    else:
        results_df.to_csv(log_file, index=False)


In [23]:
evaluate(model, test_loader, device, fold=6, model_name="SwinTransformer_Test", epoch=1)


Avaliação do modelo SwinTransformer_Test - Fold 6 - Epoch 1
Accuracy: 96.92%
Precision: 0.9695 | Recall: 0.9688 | F1: 0.9691
TP: 2670 | FP: 84 | TN: 2672 | FN: 86

 Classification Report:

              precision    recall  f1-score   support

           0     0.9688    0.9695    0.9692      2756
           1     0.9695    0.9688    0.9691      2756

    accuracy                         0.9692      5512
   macro avg     0.9692    0.9692    0.9692      5512
weighted avg     0.9692    0.9692    0.9692      5512



### Usando optuna

In [16]:
def train(
    model,
    train_loader,
    val_loader,
    epochs=5,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    model_name='SwinTransformer',
    fold=0,
    criterion=None,
    optimizer=None,
    scheduler=None
):
    model.to(device)

    # Define defaults se não forem passados
    if criterion is None:
        criterion = nn.CrossEntropyLoss()
    if optimizer is None:
        optimizer = optim.Adam(model.parameters(), lr=1e-4)
    if scheduler is None:
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

    log_file = "kfold_epoch_results.csv"
    results_list = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        current_lr = optimizer.param_groups[0]['lr']

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        scheduler.step()

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}, LR: {current_lr:.6f}")

        # Validação
        model.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        accuracy = 100 * np.mean(np.array(all_preds) == np.array(all_labels))
        precision = precision_score(all_labels, all_preds, zero_division=0)
        recall = recall_score(all_labels, all_preds, zero_division=0)
        f1 = f1_score(all_labels, all_preds, zero_division=0)
        cm = confusion_matrix(all_labels, all_preds)
        tn, fp, fn, tp = cm.ravel()

        results_list.append({
            "fold": fold,
            "epoch": epoch + 1,
            "model": model_name,
            "loss": running_loss / len(train_loader),
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "true_positives": tp,
            "false_positives": fp,
            "true_negatives": tn,
            "false_negatives": fn,
            "learning_rate": current_lr
        })

    torch.save(model.state_dict(), f"swin_fold{fold}.pth")

    results_df = pd.DataFrame(results_list)
    if os.path.exists(log_file):
        results_df.to_csv(log_file, mode='a', header=False, index=False)
    else:
        results_df.to_csv(log_file, index=False)


In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset, DataLoader
import numpy as np
import os
from sklearn.model_selection import train_test_split, StratifiedKFold

# 1. Extrai os rótulos do dataset original
labels = [dataset[i][1] for i in range(len(dataset))]

# 2. Divide em 80% treino+validação e 20% teste
train_val_indices, test_indices = train_test_split(
    list(range(len(labels))),
    test_size=0.2,
    stratify=labels,
    random_state=42
)

# 3. Cria os Subsets
train_val_dataset = Subset(dataset, train_val_indices)
test_dataset = Subset(dataset, test_indices)

# 4. Cria o DataLoader do conjunto de teste (fixo)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 5. Prepara os rótulos de treino/validação para o StratifiedKFold
train_val_labels = [dataset[i][1] for i in train_val_indices]

# 6. Configura o K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 7. Inicia o loop de treinamento por fold
for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(train_val_dataset)), train_val_labels)):
    print(f"\n----- Fold {fold + 1} -----")

    train_subset = Subset(train_val_dataset, train_idx)
    val_subset = Subset(train_val_dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)

    model = get_swin_model()
    model_name = f"SwinTransformer_Fold{fold+1}"

    if fold == 1:  # Apenas no Fold 2
        # Hiperparâmetros do Optuna
        lr = 9.597089396219724e-06
        weight_decay = 1.682757828554389e-06
        criterion = nn.CrossEntropyLoss()

        # Caminho para os pesos do fold 2
        model_path = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\swin-variations\swin-sem-optuna\modelos_salvos_swin\swin_fold2.pth"

        # Carrega os pesos salvos
        if os.path.exists(model_path):
            print(f"Carregando pesos do modelo do caminho:\n{model_path}")
            model.load_state_dict(torch.load(model_path, map_location='cuda' if torch.cuda.is_available() else 'cpu'))
        else:
            raise FileNotFoundError(f"Arquivo de pesos não encontrado: {model_path}")

        # Define otimizador e scheduler com Optuna
        optimizer = optim.RMSprop(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

        # Continua o treinamento com os novos parâmetros
        train(
            model, train_loader, val_loader,
            model_name=model_name,
            epochs=20,
            fold=fold + 1,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler
        )
    else:
        # Treinamento normal para os demais folds
        train(model, train_loader, val_loader, model_name=model_name, epochs=20, fold=fold+1)



----- Fold 1 -----
Epoch [1/20], Loss: 0.1444, LR: 0.000100
Epoch [2/20], Loss: 0.0985, LR: 0.000100
Epoch [3/20], Loss: 0.0837, LR: 0.000100
Epoch [4/20], Loss: 0.0723, LR: 0.000100
Epoch [5/20], Loss: 0.0575, LR: 0.000100
Epoch [6/20], Loss: 0.0501, LR: 0.000100
Epoch [7/20], Loss: 0.0423, LR: 0.000100
Epoch [8/20], Loss: 0.0399, LR: 0.000100
Epoch [9/20], Loss: 0.0333, LR: 0.000100
Epoch [10/20], Loss: 0.0237, LR: 0.000100
Epoch [11/20], Loss: 0.0108, LR: 0.000010
Epoch [12/20], Loss: 0.0055, LR: 0.000010
Epoch [13/20], Loss: 0.0034, LR: 0.000010
Epoch [14/20], Loss: 0.0028, LR: 0.000010
Epoch [15/20], Loss: 0.0040, LR: 0.000010
Epoch [16/20], Loss: 0.0029, LR: 0.000010
Epoch [17/20], Loss: 0.0031, LR: 0.000010
Epoch [18/20], Loss: 0.0018, LR: 0.000010
Epoch [19/20], Loss: 0.0022, LR: 0.000010
Epoch [20/20], Loss: 0.0027, LR: 0.000010

----- Fold 2 -----
Carregando pesos do modelo do caminho:
C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\swin-variations\swin-sem-optuna\mode

C:\Users\sthem\AppData\Local\Temp\ipykernel_27304\2621758701.py:58: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location='

Epoch [1/20], Loss: 0.0412, LR: 0.000010
Epoch [2/20], Loss: 0.0292, LR: 0.000010
Epoch [3/20], Loss: 0.0212, LR: 0.000010
Epoch [4/20], Loss: 0.0155, LR: 0.000010
Epoch [5/20], Loss: 0.0107, LR: 0.000010
Epoch [6/20], Loss: 0.0060, LR: 0.000010
Epoch [7/20], Loss: 0.0053, LR: 0.000010
Epoch [8/20], Loss: 0.0041, LR: 0.000010
Epoch [9/20], Loss: 0.0055, LR: 0.000010
Epoch [10/20], Loss: 0.0043, LR: 0.000010
Epoch [11/20], Loss: 0.0033, LR: 0.000001
Epoch [12/20], Loss: 0.0032, LR: 0.000001
Epoch [13/20], Loss: 0.0033, LR: 0.000001
Epoch [14/20], Loss: 0.0041, LR: 0.000001
Epoch [15/20], Loss: 0.0017, LR: 0.000001
Epoch [16/20], Loss: 0.0031, LR: 0.000001
Epoch [17/20], Loss: 0.0026, LR: 0.000001
Epoch [18/20], Loss: 0.0033, LR: 0.000001
Epoch [19/20], Loss: 0.0027, LR: 0.000001
Epoch [20/20], Loss: 0.0026, LR: 0.000001

----- Fold 3 -----
Epoch [1/20], Loss: 0.1348, LR: 0.000100
Epoch [2/20], Loss: 0.0995, LR: 0.000100
Epoch [3/20], Loss: 0.0782, LR: 0.000100
Epoch [4/20], Loss: 0.0714